## Project: End-to-End Customer Intelligence & Revenue Optimization System

## Phase 1 — Data Layer
#### Use real-world dataset (e-commerce, SaaS, telecom churn)
        Build a clean feature pipeline
        Handle missing data
        Build cohort features
        RFM analysis

In [94]:
import pandas as pd
import numpy as np
import streamlit as st
from fastapi import FastAPI
import uvicorn
import matplotlib.pyplot as plt
import seaborn as sns

In [95]:
orders = pd.read_csv('C:/Users/MIKEYkun/Downloads/OList Dataset/olist_orders_dataset.csv')
order_items = pd.read_csv('C:/Users/MIKEYkun/Downloads/OList Dataset/olist_order_items_dataset.csv')
payments = pd.read_csv('C:/Users/MIKEYkun/Downloads/OList Dataset/olist_order_payments_dataset.csv')
customers = pd.read_csv('C:/Users/MIKEYkun/Downloads/OList Dataset/olist_customers_dataset.csv')
products = pd.read_csv('C:/Users/MIKEYkun/Downloads/OList Dataset/olist_products_dataset.csv')
reviews = pd.read_csv('C:/Users/MIKEYkun/Downloads/OList Dataset/olist_order_reviews_dataset.csv')
sellers = pd.read_csv('C:/Users/MIKEYkun/Downloads/OList Dataset/olist_sellers_dataset.csv')
geo = pd.read_csv('C:/Users/MIKEYkun/Downloads/OList Dataset/olist_geolocation_dataset.csv')
category = pd.read_csv('C:/Users/MIKEYkun/Downloads/OList Dataset/product_category_name_translation.csv')

In [96]:
orders.info()
orders.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

In [97]:
orders['order_purchase_timestamp'] = pd.to_datetime(
    orders['order_purchase_timestamp']
)

In [98]:
df = orders.merge(customers, on='customer_id', how='left')
df = df.merge(order_items, on='order_id', how='left')
df = df.merge(payments, on='order_id', how='left')
df = df.merge(products, on='product_id', how='left')

In [99]:
df = df[df['order_status'] == 'delivered']
df['Revenue'] = df['price'] + df['freight_value']

In [100]:
df = df.merge(reviews[['order_id', 'review_score']],
              on='order_id',
              how='left')

In [101]:
df.columns

Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date',
       'customer_unique_id', 'customer_zip_code_prefix', 'customer_city',
       'customer_state', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value', 'payment_sequential',
       'payment_type', 'payment_installments', 'payment_value',
       'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',
       'product_length_cm', 'product_height_cm', 'product_width_cm', 'Revenue',
       'review_score'],
      dtype='object')

In [102]:
df['product_category_name'] = df['product_category_name'].fillna("unknown")
df['review_score'] = df['review_score'].fillna(df['review_score'].median())

In [103]:
df = df.dropna(subset=['customer_unique_id'])

In [104]:
snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

customer_df = df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'Revenue': 'sum',
    'review_score': 'mean'
}).reset_index()

In [105]:
customer_df.columns = [
    'CustomerID',
    'Recency',
    'Frequency',
    'Monetary',
    'AvgReviewScore'
]

In [106]:
df['OrderMonth'] = df['order_purchase_timestamp'].dt.to_period('M')
df['CohortMonth'] = df.groupby('customer_unique_id')['OrderMonth'].transform('min')

In [107]:
df['CohortIndex'] = (df['OrderMonth'] - df['CohortMonth']).apply(lambda x: x.n)

In [108]:
cohort_data = df.groupby(['CohortMonth', 'CohortIndex'])['customer_unique_id'] \
                .nunique() \
                .reset_index()

cohort_table = cohort_data.pivot(
    index='CohortMonth',
    columns='CohortIndex',
    values='customer_unique_id'
)

In [109]:
rfm = customer_df.copy()
rfm['R_score'] = pd.qcut(
    rfm['Recency'],
    5,
    labels=[5,4,3,2,1]
)

rfm['F_score'] = pd.qcut(
    rfm['Frequency'].rank(method='first'),
    5,
    labels=[1,2,3,4,5]
)

rfm['M_score'] = pd.qcut(
    rfm['Monetary'].rank(method='first'),
    5,
    labels=[1,2,3,4,5]
)

rfm['RFM_Score'] = (
    rfm['R_score'].astype(str) +
    rfm['F_score'].astype(str) +
    rfm['M_score'].astype(str)
)

In [110]:
rfm[['R_score','F_score','M_score']] = rfm[['R_score','F_score','M_score']].astype(int)

In [111]:
def segment_customer(row):
    
    if row['R_score'] >= 4 and row['F_score'] >= 4 and row['M_score'] >= 4:
        return "Champions 🔥"
    
    elif row['R_score'] >= 3 and row['F_score'] >= 3:
        return "Loyal Customers 💎"
    
    elif row['R_score'] >= 4 and row['F_score'] <= 2:
        return "New Customers 🆕"
    
    elif row['R_score'] <= 2 and row['F_score'] >= 3:
        return "At Risk ⚠️"
    
    elif row['R_score'] <= 2 and row['F_score'] <= 2:
        return "Lost Customers ❌"
    
    else:
        return "Potential Loyalist ⭐"

In [112]:
rfm['Segment'] = rfm.apply(segment_customer, axis=1)

In [113]:
rfm['Segment'].value_counts()

Segment
Loyal Customers 💎       27322
At Risk ⚠️              22230
Lost Customers ❌        14986
New Customers 🆕         14984
Potential Loyalist ⭐     7373
Champions 🔥              6463
Name: count, dtype: int64

## Phase 2 — Modeling
###  Churn Prediction (Classification)
        Logistic Regression
        Random Forest
        XGBoost
        Precision/Recall tuning (business tradeoffs)

###  LTV Prediction (Regression)
        Gradient boosting
        Evaluate with RMSE + business metrics

###  Recommendation System
        Collaborative filtering
        Matrix factorization OR
        Hybrid model

### PART 1 — Churn Prediction

In [114]:
cutoff_date = df['order_purchase_timestamp'].max() - pd.Timedelta(days=90)

In [115]:
train_df = df[df['order_purchase_timestamp'] <= cutoff_date]
future_df = df[df['order_purchase_timestamp'] > cutoff_date]

In [116]:
snapshot_date = train_df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

customer_features = train_df.groupby('customer_unique_id').agg({
    'order_purchase_timestamp': lambda x: (snapshot_date - x.max()).days,
    'order_id': 'nunique',
    'Revenue': 'sum',
    'review_score': 'mean',
    'freight_value': 'mean'
}).reset_index()

customer_features.columns = [
    'CustomerID',
    'Recency',
    'Frequency',
    'Monetary',
    'AvgReviewScore',
    'AvgFreight'
]

In [117]:
customer_features['AOV'] = (
    customer_features['Monetary'] /
    customer_features['Frequency']
)

In [118]:
customer_features['RepeatBuyer'] = np.where(
    customer_features['Frequency'] > 1, 1, 0
)

In [119]:
future_customers = future_df['customer_unique_id'].unique()

customer_features['Churn'] = np.where(
    customer_features['CustomerID'].isin(future_customers),
    0,  # active
    1   # churned
)

In [120]:
customer_features['Churn'].value_counts(normalize=True)

Churn
1    0.994411
0    0.005589
Name: proportion, dtype: float64

In [121]:
from sklearn.utils import resample
from sklearn.linear_model import LogisticRegression

In [122]:
X = customer_features[['Recency', 'Frequency', 'Monetary', 'AvgReviewScore', 'AOV', 'RepeatBuyer', 'AvgFreight']]
y = customer_features['Churn']

In [123]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [124]:
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [125]:
y_pred = lr.predict(X_test)

In [126]:
from sklearn.metrics import classification_report, roc_auc_score

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr.predict_proba(X_test)[:,1]))

              precision    recall  f1-score   support

           0       0.00      0.00      0.00        84
           1       0.99      1.00      1.00     14980

    accuracy                           0.99     15064
   macro avg       0.50      0.50      0.50     15064
weighted avg       0.99      0.99      0.99     15064

ROC-AUC: 0.6095695053722423


C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [127]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42
)
rf.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",8
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_

In [128]:
from xgboost import XGBClassifier

In [129]:
xgb = XGBClassifier(
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss'
)

xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [130]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1])

0.5249221183800623

In [165]:
from sklearn.metrics import classification_report, roc_auc_score

# Logistic Regression
print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, lr.predict_proba(X_test)[:,1]))

=== Logistic Regression ===
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        84
           1       0.99      1.00      1.00     14980

    accuracy                           0.99     15064
   macro avg       0.50      0.50      0.50     15064
weighted avg       0.99      0.99      0.99     15064

ROC-AUC: 0.6095695053722423


C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [166]:
# Random Forest
print("=== Random Forest ===")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))

=== Random Forest ===
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        84
           1       0.99      1.00      1.00     14980

    accuracy                           0.99     15064
   macro avg       0.50      0.50      0.50     15064
weighted avg       0.99      0.99      0.99     15064



C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


ROC-AUC: 0.549544233581283


In [167]:
# XGBoost
print("=== XGBoost ===")
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]))

=== XGBoost ===
              precision    recall  f1-score   support

           0       0.00      0.00      0.00        84
           1       0.99      1.00      1.00     14980

    accuracy                           0.99     15064
   macro avg       0.50      0.50      0.50     15064
weighted avg       0.99      0.99      0.99     15064

ROC-AUC: 0.5249221183800623


C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\MIKEYkun\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [168]:
y_prob = xgb.predict_proba(X_test)[:,1]

threshold = 0.35  # tuned
y_pred_custom = (y_prob > threshold).astype(int)

###  PART 2 — LTV PREDICTION (SERIOUS VERSION)

In [133]:
future_revenue = future_df.groupby('customer_unique_id')['Revenue'].sum().reset_index()
future_revenue.columns = ['CustomerID', 'FutureRevenue']

ltv_df = customer_features.merge(
    future_revenue,
    left_on='CustomerID',
    right_on='CustomerID',
    how='left'
).fillna(0)

In [134]:
X = ltv_df[['Recency', 'Frequency', 'Monetary', 'AvgReviewScore', 'AvgFreight', 'AOV', 'RepeatBuyer']]
y = ltv_df['FutureRevenue']

In [135]:
X_train_ltv, X_test_ltv, y_train_ltv, y_test_ltv = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [136]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.05
)

gbr.fit(X_train, y_train)

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.05
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",500
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",5
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft

In [137]:
from sklearn.metrics import mean_squared_error, r2_score

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

In [138]:
print("RMSE:", rmse)
print("R2 Score:", r2)

RMSE: 0.07467401273829244
R2 Score: -0.00560747663551453


###  PART 3 — RECOMMENDATION SYSTEM

In [139]:
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity

In [140]:
interaction = df.groupby(['customer_unique_id', 'product_id'])['order_id'].count().reset_index()
interaction.rename(columns={'order_id':'purchase_count'}, inplace=True)

In [141]:
customer_map = {id:i for i,id in enumerate(interaction['customer_unique_id'].unique())}
product_map = {id:i for i,id in enumerate(interaction['product_id'].unique())}
interaction['cust_idx'] = interaction['customer_unique_id'].map(customer_map)
interaction['prod_idx'] = interaction['product_id'].map(product_map)

In [142]:
sparse_matrix = csr_matrix((interaction['purchase_count'], 
                            (interaction['cust_idx'], interaction['prod_idx'])))

In [143]:
svd = TruncatedSVD(n_components=20, random_state=42)
latent_matrix = svd.fit_transform(sparse_matrix)

In [144]:
def get_top_similar_customers(cust_id, top_n=10):
    idx = customer_map[cust_id]
    cust_vec = latent_matrix[idx].reshape(1, -1)
    sim_scores = cosine_similarity(cust_vec, latent_matrix)[0]
    top_indices = np.argsort(sim_scores)[::-1][1:top_n+1]  # exclude self
    rev_map = {v:k for k,v in customer_map.items()}
    return [rev_map[i] for i in top_indices]

In [145]:
rfm['RFM_weight'] = rfm['R_score'].astype(int)*0.4 + rfm['F_score'].astype(int)*0.4 + rfm['M_score'].astype(int)*0.2
rfm_dict = rfm.set_index('CustomerID')['RFM_weight'].to_dict()

In [146]:
popularity = interaction.groupby('prod_idx')['purchase_count'].sum()
popularity = popularity / popularity.max()
popularity_dict = popularity.to_dict()

In [147]:
interaction['customer_unique_id'].head(5)

0    0000366f3b9a7992bf8c76cfdf3221e2
1    0000b849f77a49e4a4ce2b2a4ca5be3f
2    0000f46a3911fa3c0805444483337064
3    0000f6ccb0745a6a4b88665a16c9f078
4    0004aac84e0df4da2b147fca70cf8255
Name: customer_unique_id, dtype: object

In [148]:
def hybrid_recommend(cust_id, top_n=10):
    similar_customers = get_top_similar_customers(cust_id, top_n=50)
    similar_idx = [customer_map[c] for c in similar_customers]
    
    # Select purchases from similar customers
    mask = interaction['cust_idx'].isin(similar_idx)
    sub = interaction[mask]
    
    product_scores = {}
    for _, row in sub.iterrows():
        pid = row['prod_idx']
        cf_score = row['purchase_count'] / sub['purchase_count'].max()
        pop_score = popularity_dict.get(pid, 0)
        rfm_score = rfm_dict.get(cust_id, 0) / max(rfm_dict.values())
        final_score = 0.5*cf_score + 0.3*pop_score + 0.2*rfm_score
        product_scores[pid] = max(product_scores.get(pid, 0), final_score)
    
    top_products_idx = sorted(product_scores.items(), key=lambda x:x[1], reverse=True)[:top_n]
    rev_product_map = {v:k for k,v in product_map.items()}
    return [rev_product_map[i[0]] for i in top_products_idx]

# Example
hybrid_recommend('0000366f3b9a7992bf8c76cfdf3221e2', top_n=10)

['372645c7439f9661fbbacfd129aa92ec',
 '525947dbe3304ac32bf51602f9557c12',
 '42155695adbe665066ad812855fe523a',
 '68a058d125ba1544e726898f0b8c0523',
 '429e7401fafb76436f15e86498bd7364',
 'eab67bf937aaadc19f83383a331d2dd9',
 '120fa011365fc39efe382cba4e50999e',
 'cc1b0e67ffb98a08c886e8c3c27a915f',
 'eb53f94fdc60278efcef123bb275658a',
 'bd1aace4fad5609f005fd721b45dcec4']

### Phase 3 — Business Simulation Layer

In [149]:
import matplotlib.pyplot as plt
import seaborn as sns

In [150]:
rfm['RFM_weight'] = rfm['R_score'].astype(int)*0.4 + \
                    rfm['F_score'].astype(int)*0.4 + \
                    rfm['M_score'].astype(int)*0.2

In [151]:
churn_prob = lr.predict_proba(X_test)[:,1]  # probability of churn
churn_df = pd.DataFrame({
        'customer_unique_id': X_test.index, 'churn_prob': churn_prob
    })
churn_df['customer_unique_id'] = churn_df['customer_unique_id'].astype(str)
rfm['CustomerID'] = rfm['CustomerID'].astype(str)

In [152]:
sim_df = churn_df.merge(
    rfm[['CustomerID','RFM_weight']],
    left_on='customer_unique_id',
    right_on='CustomerID',
    how='left'
)
sim_df.drop(columns=['CustomerID'], inplace=True)
sim_df.head()

,customer_unique_id,churn_prob,RFM_weight
0,6601,0.996378,NaN
1,50641,0.992157,NaN
2,3465,0.995318,NaN
3,72535,0.996545,NaN
4,8115,0.994412,NaN


In [153]:
churn_threshold = 0.6  
high_risk = sim_df[sim_df['churn_prob'] > churn_threshold].copy()
print(f"Number of high-risk customers: {len(high_risk)}")
high_risk.head()

Number of high-risk customers: 15064


,customer_unique_id,churn_prob,RFM_weight
0,6601,0.996378,NaN
1,50641,0.992157,NaN
2,3465,0.995318,NaN
3,72535,0.996545,NaN
4,8115,0.994412,NaN


In [154]:
high_risk['churn_prob_after_offer'] = high_risk['churn_prob'] * (1 - retention_lift)

In [155]:
orders_items = orders.merge(order_items, on="order_id", how="left")

customer_revenue = orders_items.groupby("customer_id")["price"].sum().reset_index()
customer_revenue.rename(columns={"price": "total_revenue"}, inplace=True)

In [156]:
# Frequency: number of orders
customer_freq = orders.groupby("customer_id")["order_id"].nunique().reset_index()
customer_freq.rename(columns={"order_id": "order_count"}, inplace=True)

In [157]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])
last_order = orders.groupby("customer_id")["order_purchase_timestamp"].max().reset_index()
last_order["recency_days"] = (orders["order_purchase_timestamp"].max() - last_order["order_purchase_timestamp"]).dt.days

In [158]:
customer_features = customer_revenue.merge(customer_freq, on="customer_id")
customer_features = customer_features.merge(last_order[["customer_id","recency_days"]], on="customer_id")

customer_features["churn_prob"] = (
    0.4 * (customer_features["recency_days"] / customer_features["recency_days"].max()) +
    0.3 * (1 - customer_features["order_count"] / customer_features["order_count"].max()) +
    0.3 * (1 - customer_features["total_revenue"] / customer_features["total_revenue"].max())
)

customer_features["churn_prob"] = customer_features["churn_prob"].clip(0,1)

df_predictions = customer_features[["customer_id", "churn_prob"]]

print(df_predictions.head())

                        customer_id  churn_prob
0  00012a2ce6f8dcda20d059ce98491703    0.472607
1  000161a058600d5901f007fab4c27140    0.536080
2  0001fd6190edaaf884bcaf3d49edf079    0.604791
3  0002414f95344307404f0ace7a26f1d5    0.517898
4  000379cdec625522490c315e70c7a9fb    0.400515


In [159]:
df_model = df_predictions.copy()  # your churn predictions DataFrame
df_sim = customer_revenue.merge(df_model, on="customer_id", how="left")
df_sim["churn_prob"] = df_sim["churn_prob"].fillna(df_sim["churn_prob"].mean())

In [160]:
discount_amount = 10  # $10 retention offer
df_sim["discount_offered"] = discount_amount

In [161]:
discount_effectiveness = 0.3
df_sim["churn_prob_after_discount"] = df_sim["churn_prob"] * (1 - discount_effectiveness)
df_sim["retention_prob"] = 1 - df_sim["churn_prob_after_discount"]
df_sim["expected_revenue"] = df_sim["retention_prob"] * df_sim["total_revenue"]
df_sim["discount_cost"] = discount_amount  # cost per user
df_sim["net_profit"] = df_sim["expected_revenue"] - df_sim["discount_cost"]

In [162]:
total_expected_revenue = df_sim["expected_revenue"].sum()
total_discount_cost = df_sim["discount_cost"].sum()
roi = (total_expected_revenue - total_discount_cost) / total_discount_cost

print(f"High-Risk Customers: {len(high_risk)}")
print(f"Total Expected Revenue: ${total_expected_revenue:.2f}")
print(f"Total Discount Cost: ${total_discount_cost:.2f}")
print(f"Expected ROI: {roi*100:.2f}%")

High-Risk Customers: 15064
Total Expected Revenue: $9405827.26
Total Discount Cost: $994410.00
Expected ROI: 845.87%


In [163]:
simulations = 1000
roi_list = []

for i in range(simulations):
    simulated_retention = df_sim["churn_prob"] * (1 - np.random.normal(discount_effectiveness, 0.05, len(df_sim)))
    simulated_retention = 1 - simulated_retention
    expected_revenue_mc = simulated_retention * df_sim["total_revenue"]
    net_profit_mc = expected_revenue_mc - discount_amount
    roi_list.append(net_profit_mc.sum() / total_discount_cost)

print(f"Monte Carlo Average ROI: {np.mean(roi_list)*100:.2f}%")

Monte Carlo Average ROI: 845.86%


In [164]:
summary = pd.DataFrame({
    "Metric": ["Total Users", "Total Expected Revenue", "Total Discount Cost", "Expected ROI (%)"],
    "Value": [len(df_sim), total_expected_revenue, total_discount_cost, roi*100]
})

summary

,Metric,Value
0,Total Users,9.944100e+04
1,Total Expected Revenue,9.405827e+06
2,Total Discount Cost,9.944100e+05
3,Expected ROI (%),8.458701e+02


In [169]:
df_predictions.to_csv("df_predictions.csv", index=False)
high_risk.to_csv("df_simulation.csv", index=False)

In [170]:
import os

print("df_predictions exists:", os.path.exists("df_predictions.csv"))
print("df_simulation exists:", os.path.exists("df_simulation.csv"))

df_predictions exists: True
df_simulation exists: True


In [171]:
os.getcwd()

'C:\\Users\\MIKEYkun\\Downloads\\OList Dataset'